# Wikipedia -> tokenizer -> tokenized `.bin` (resumable, RAM-safe, saves to Google Drive)

**Run this on Google Colab (CPU runtime is enough).** Colab: Runtime -> Change runtime type -> **CPU**. No GPU needed for this data-prep step.

### What changed from the version that crashed
The old notebook OOM'd because of a few things that quietly pile up RAM on Colab's free tier (~12 GB):
1. `datasets` streaming `.shuffle(buffer_size=20_000)` keeps 20,000 full articles resident just for shuffling — on English Wikipedia that buffer alone can be several GB.
2. Articles were batched up to 1000-at-a-time before calling `tok.encode_batch(...)`, so up to 1000 full articles + their encodings sat in RAM simultaneously, and the Python list was never actively freed.
3. At the end, `np.memmap(...).max()` and `tok.decode(sample)` on 200k tokens caused the whole file to get paged into RAM for the scan, and nothing was chunked.
4. No periodic `gc.collect()`, so short-lived garbage (batches, encodings) accumulated between Python's GC cycles.

This version fixes all four: smaller shuffle buffer, small fixed-size micro-batches that are processed and dropped immediately, explicit `gc.collect()` on every checkpoint, chunked stats at the end instead of loading the full memmap into a Python computation, and it prints live RSS (process RAM) via `psutil` so you can see memory stay flat instead of climbing.

**How checkpointing works:** every `CKPT_MINUTES` (10) minutes the notebook closes the current shard file (`train_00007.bin`, `val_00007.bin`), copies it to Drive, verifies the size, and only then updates `state.json` on Drive. If Colab disconnects, just re-run all cells: it reads `state.json` and continues from the last checkpoint. You lose at most ~10 minutes of work.

Start with `TEST_RUN = True` (about 10-15 min) to check everything, then set it to `False`.


In [ ]:
!pip install -q -U datasets tokenizers psutil

In [ ]:
import os, re, sys, json, time, shutil, glob, gc
import numpy as np
import psutil
from tqdm.auto import tqdm
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["HF_DATASETS_IN_MEMORY_MAX_SIZE"] = "0"  # never let HF cache a split in RAM

_proc = psutil.Process()
def ram_mb():
    return _proc.memory_info().rss / 2**20

def log_ram(tag=""):
    print(f"[ram] {ram_mb():,.0f} MB{'  ' + tag if tag else ''}")

# ------------------------- CONFIG -------------------------
TEST_RUN = True        # <- set to False for the real run
CKPT_MINUTES = 10       # checkpoint to Drive this often

DATASET = "wikimedia/wikipedia"
SUBSET = "20231101.en"
VOCAB_SIZE = 32768
MIN_CHARS = 1000        # drop stubs / very short articles
VAL_EVERY = 200         # every 200th article goes to validation (~0.5%)
SEED = 1337

# RAM-safety knobs. Keep these small: streaming from HF + Drive I/O is the
# bottleneck anyway, not CPU, so there is no speed benefit to bigger buffers.
SHUFFLE_BUFFER = 1_000   # articles held for shuffling (was 20,000, then 5,000)
MICRO_BATCH = 512        # articles per encode_batch() call (was 1000, then 128)
GC_EVERY_BATCHES = 20    # force a garbage-collection sweep this often

if TEST_RUN:
    TOKENIZER_ARTICLES = 20_000
    TARGET_TOKENS = 30_000_000
else:
    TOKENIZER_ARTICLES = 100_000     # ~500 MB of text is plenty for a 16k-vocab BPE
    TARGET_TOKENS = 1_000_000_000    # training tokens (val is extra, ~0.5%)

# ------------------- WHERE THINGS ARE SAVED -------------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/slm_wiki"   # persistent (Google Drive)
    WORK_DIR = "/content/work"                     # fast local scratch
else:
    SAVE_DIR = "/kaggle/working/slm_wiki"
    WORK_DIR = "/kaggle/temp/work"
if TEST_RUN:
    SAVE_DIR += "_test"        # keep test files separate from the real run
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
STATE_PATH = f"{SAVE_DIR}/state.json"
print("=" * 60)
print(f"TEST_RUN = {TEST_RUN}")
print(f"target tokens = {TARGET_TOKENS:,}")
print(f"saving to = {SAVE_DIR}")
print("=" * 60)
if not TEST_RUN:
    assert TARGET_TOKENS == 1_000_000_000, "TEST_RUN is False but TARGET_TOKENS looks like a test value - restart the runtime and run all cells top to bottom"
    assert not SAVE_DIR.endswith("_test"), "TEST_RUN is False but SAVE_DIR still points at the _test folder - restart the runtime and run all cells top to bottom"
    print(">>> CONFIRMED: this is the FULL run, saving to", SAVE_DIR)
else:
    print(">>> This is the TEST run (30M tokens). Set TEST_RUN = False and RESTART THE RUNTIME to do the full run.")
log_ram("startup")

In [ ]:
# Light cleaning. The HF dataset is already plain text (no wiki markup),
# so we only drop trailing boilerplate sections and normalise blank lines.
STOP_SECTIONS = {"references", "external links", "see also", "further reading",
                 "footnotes", "bibliography"}

def clean_text(t):
    out = []
    for ln in t.split("\n"):
        if ln.strip().lower() in STOP_SECTIONS:
            break
        out.append(ln)
    t = "\n".join(out)
    t = re.sub(r"\n{3,}", "\n\n", t).strip()
    return t

def raw_stream(seed, skip=0):
    # Deterministic for a given seed, so a resumed run sees the same order.
    # A small buffer_size matters for a second reason beyond RAM: the shuffle
    # buffer has to be *filled* (i.e. that many articles downloaded/decoded)
    # before the dataset starts yielding anything, and HF streams the
    # underlying Parquet shards over the network. A big buffer on a slow
    # free-tier connection means several minutes where progress looks stuck
    # at 0% while nothing has been tokenized yet - it isn't RAM or CPU bound,
    # it's the network fill. 1,000 fills fast and shuffles enough for our
    # purposes (a random 1000-article local window over an already
    # topically-scattered Wikipedia dump).
    ds = load_dataset(DATASET, SUBSET, split="train", streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=SHUFFLE_BUFFER)
    if skip:
        ds = ds.skip(skip)
    for ex in ds:
        yield ex["text"]

def article_stream(seed):
    for text in raw_stream(seed):
        t = clean_text(text)
        if len(t) >= MIN_CHARS:
            yield t

# Look at a few samples to make sure the cleaning is sane
for i, t in enumerate(article_stream(SEED)):
    print(f"--- sample {i} ({len(t)} chars) ---")
    print(t[:400].replace("\n", " / "))
    print("...", t[-200:].replace("\n", " / "))
    if i == 2:
        break
del t
log_ram("after sample check")

## Step 1: train the tokenizer
Byte-level BPE (like GPT-2), 16k vocab. Smaller vocab = more of your parameter budget goes to the transformer layers instead of embeddings.

This step still has to hold `TOKENIZER_ARTICLES` worth of text at once (the underlying Rust trainer needs the corpus), but it now consumes it in small `MICRO_BATCH`-sized chunks fed through a generator instead of one 1000-article batch, and drops each chunk immediately after handing it to the trainer. The result is saved to Drive, and if `tokenizer.json` already exists the step is skipped entirely — so a crash here never repeats the work.

In [ ]:
TOK_PATH = f"{SAVE_DIR}/tokenizer.json"

if os.path.exists(TOK_PATH):
    tok = Tokenizer.from_file(TOK_PATH)
    print("loaded existing tokenizer from Drive, vocab =", tok.get_vocab_size())
else:
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=["<|endoftext|>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        min_frequency=2,
        show_progress=True,
    )

    def batch_iter(n_articles, bs=MICRO_BATCH):
        batch = []
        for i, t in enumerate(tqdm(article_stream(SEED), total=n_articles, desc="tokenizer data")):
            if i >= n_articles:
                break
            batch.append(t)
            if len(batch) == bs:
                yield batch
                batch = []
            if i % (bs * GC_EVERY_BATCHES) == 0:
                log_ram(f"tokenizer data @ article {i}")
        if batch:
            yield batch

    t0 = time.time()
    tok.train_from_iterator(batch_iter(TOKENIZER_ARTICLES), trainer=trainer)
    tok.save(TOK_PATH)
    gc.collect()
    print(f"tokenizer trained in {(time.time()-t0)/60:.1f} min, vocab size = {tok.get_vocab_size()}")

log_ram("after tokenizer training")
s = "The Eiffel Tower is a wrought-iron lattice tower in Paris, France."
enc = tok.encode(s)
print(enc.tokens)
print("roundtrip ok:", tok.decode(enc.ids) == s)

## Step 2: tokenize in checkpointed shards
Each article is followed by `<|endoftext|>`. Tokens are `uint16`. Progress is saved to Drive every `CKPT_MINUTES`.

**RAM fix:** articles are now flushed to `tok.encode_batch()` every `MICRO_BATCH` (128) articles instead of 1000, the batch list is reassigned (not just emptied) so the old list is immediately collectible, and `gc.collect()` runs on every shard checkpoint. RSS is logged at each checkpoint so you can watch it stay flat over a multi-hour run.

**If Colab disconnects: reconnect, re-run the cells from the top, and this cell resumes automatically.** (On resume it has to re-stream the articles it already processed to get back to the right spot, which costs some minutes but no re-tokenizing.)

In [ ]:
EOT = tok.token_to_id("<|endoftext|>")
assert tok.get_vocab_size() < 65536

def load_state():
    if os.path.exists(STATE_PATH):
        return json.load(open(STATE_PATH))
    return dict(raw_seen=0, n_articles=0, train_tokens=0, val_tokens=0,
                next_shard=0, done=False)

def save_state(st):
    tmp = STATE_PATH + ".tmp"
    json.dump(st, open(tmp, "w"))
    os.replace(tmp, STATE_PATH)          # atomic: state.json is never half-written

def open_shard(k):
    return (open(f"{WORK_DIR}/train_{k:05d}.bin", "wb"),
            open(f"{WORK_DIR}/val_{k:05d}.bin", "wb"))

def write_batch(buf, st, train_f, val_f):
    for e in tok.encode_batch(buf):
        ids = np.array(e.ids + [EOT], dtype=np.uint16)
        if st["n_articles"] % VAL_EVERY == 0:
            val_f.write(ids.tobytes());   st["val_tokens"] += len(ids)
        else:
            train_f.write(ids.tobytes()); st["train_tokens"] += len(ids)
        st["n_articles"] += 1

def commit_shard(k, train_f, val_f, st):
    train_f.close(); val_f.close()
    for name in (f"train_{k:05d}.bin", f"val_{k:05d}.bin"):
        src, dst = f"{WORK_DIR}/{name}", f"{SAVE_DIR}/{name}"
        shutil.copy2(src, dst)
        assert os.path.getsize(src) == os.path.getsize(dst), f"copy of {name} incomplete"
        os.remove(src)                    # free local scratch space immediately
    st["next_shard"] = k + 1
    save_state(st)                        # only after both shard files are safely on Drive
    gc.collect()

def run():
    st = load_state()
    if st["done"]:
        print("already finished:", st)
        if TEST_RUN and st["train_tokens"] > 40_000_000:
            print("WARNING: TEST_RUN=True but state.json has", f"{st['train_tokens']:,}",
                  "tokens - this looks like leftover state from a real run in the wrong folder.")
        if not TEST_RUN and st["train_tokens"] < 40_000_000:
            print("WARNING: TEST_RUN=False but state.json only has", f"{st['train_tokens']:,}",
                  "tokens (looks like a test run). This SAVE_DIR (", SAVE_DIR, ") already has a",
                  "'done' state.json from a previous run, so Step 2 is exiting immediately without",
                  "doing the full 1B-token run. To force the real run: delete or rename",
                  f"'{STATE_PATH}' on Drive, then re-run this cell.")
        return st
    k = st["next_shard"]
    if st["raw_seen"]:
        print(f"RESUMING from shard {k}: {st['train_tokens']:,} train tokens done, "
              f"re-streaming {st['raw_seen']:,} articles to get back to the right spot...")
    train_f, val_f = open_shard(k)
    buf, last_ckpt = [], time.time()
    batches_since_gc = 0
    pbar = tqdm(total=TARGET_TOKENS, initial=st["train_tokens"], unit="tok", desc="tokenizing")
    try:
        for text in raw_stream(SEED + 1, skip=st["raw_seen"]):
            st["raw_seen"] += 1
            t = clean_text(text)
            if len(t) < MIN_CHARS:
                continue
            buf.append(t)
            if len(buf) < MICRO_BATCH:
                continue
            before = st["train_tokens"]
            write_batch(buf, st, train_f, val_f)
            buf = []                       # reassign, don't clear in place: old list becomes garbage now
            batches_since_gc += 1
            if batches_since_gc >= GC_EVERY_BATCHES:
                gc.collect()
                batches_since_gc = 0
            pbar.update(st["train_tokens"] - before)
            finished = st["train_tokens"] >= TARGET_TOKENS
            if finished or time.time() - last_ckpt >= CKPT_MINUTES * 60:
                commit_shard(k, train_f, val_f, st)
                last_ckpt = time.time()
                print(f"[checkpoint] shard {k} saved | train tokens {st['train_tokens']:,} | "
                      f"articles {st['n_articles']:,}")
                log_ram(f"after checkpoint {k}")
                if finished:
                    st["done"] = True; save_state(st)
                    return st
                k += 1
                train_f, val_f = open_shard(k)
        # stream ended before reaching the target: flush what is left
        if buf:
            write_batch(buf, st, train_f, val_f)
        commit_shard(k, train_f, val_f, st)
        st["done"] = True; save_state(st)
        return st
    finally:
        pbar.close()

t0 = time.time()
state = run()
print(json.dumps(state, indent=2))
print(f"this session: {(time.time()-t0)/60:.1f} min")
log_ram("end of step 2")

## Step 3: stitch shards into `train.bin` / `val.bin` and sanity-check
The concatenation step streams shard-to-shard with `shutil.copyfileobj` (already RAM-safe, unchanged). What changed is the **verification below it**: instead of calling `.max()` and `.decode()` on data that forces the whole memmap into the page cache at once, we scan the train file in fixed-size chunks and only decode a small fixed sample.

In [ ]:
def concat(prefix, n_shards):
    names = [f"{SAVE_DIR}/{prefix}_{k:05d}.bin" for k in range(n_shards)]
    missing = [n for n in names if not os.path.exists(n)]
    assert not missing, f"missing shards: {missing[:3]}"
    out = f"{WORK_DIR}/{prefix}.bin"
    with open(out, "wb") as w:
        for n in names:
            with open(n, "rb") as r:
                shutil.copyfileobj(r, w, 16 * 2**20)
    return out

state = load_state()
assert state["done"], "tokenizing is not finished yet - re-run the cell above"
train_path = concat("train", state["next_shard"])
val_path = concat("val", state["next_shard"])
log_ram("after concat")

train = np.memmap(train_path, dtype=np.uint16, mode="r")
val = np.memmap(val_path, dtype=np.uint16, mode="r")
assert len(train) == state["train_tokens"] and len(val) == state["val_tokens"], "token counts do not match state"
print("train tokens:", f"{len(train):,}", "| val tokens:", f"{len(val):,}")

# Chunked max-scan instead of train.max(), which would pull the whole file into RAM at once.
CHUNK = 20_000_000
max_id, eot_count = 0, 0
for i in range(0, len(train), CHUNK):
    chunk = train[i:i + CHUNK]
    max_id = max(max_id, int(chunk.max()))
    eot_count += int((chunk == EOT).sum())
    del chunk
print("max token id:", max_id, "(must be <", tok.get_vocab_size(), ")")
print("avg tokens/article:", len(train) // max(1, eot_count))

sample = train[:200_000].tolist()
print("chars/token (higher = better tokenizer):", round(len(tok.decode(sample)) / len(sample), 2))
print("\n--- decoded start of train.bin ---")
print(tok.decode(train[:300].tolist()))
del sample
log_ram("after sanity check")

In [ ]:
# Copy the final files to Drive too (uses ~2x space on Drive until you delete the shards)
for p in (train_path, val_path):
    shutil.copy2(p, f"{SAVE_DIR}/{os.path.basename(p)}")
meta = dict(dataset=f"{DATASET}/{SUBSET}", vocab_size=tok.get_vocab_size(), eot_id=EOT,
            dtype="uint16", n_articles=state["n_articles"], train_tokens=state["train_tokens"],
            val_tokens=state["val_tokens"], min_chars=MIN_CHARS, test_run=TEST_RUN)
json.dump(meta, open(f"{SAVE_DIR}/meta.json", "w"), indent=2)
print("Files in", SAVE_DIR)
for f in sorted(os.listdir(SAVE_DIR)):
    if not f.startswith(("train_0", "val_0")):
        print(" ", f, os.path.getsize(f"{SAVE_DIR}/{f}") // 2**20, "MB")
print("\nYou can now delete the train_0*.bin / val_0*.bin shards on Drive to free space.")

## Next steps
- If `TEST_RUN` looked fine and RAM (see the `[ram]` logs) stayed flat: set `TEST_RUN = False`, run all again. The real run is roughly 1-3 hours, mostly HF download + tokenizing; Drive keeps it safe if Colab drops.
- If RAM still climbs on the full run, lower `MICRO_BATCH` and `SHUFFLE_BUFFER` further (e.g. 64 and 2,000) — they trade a little throughput for a flatter memory curve.
- Check the printed samples for junk before the full run. If you see a recurring pattern, add it to `clean_text`.
- Later, load `train.bin` / `val.bin` / `tokenizer.json` from Drive in your training environment. This data-prep step needs no GPU; save your Colab GPU hours/VRAM for the actual model-training notebook.
